# Evidenze per Continual Learning nel forecasting PV

Questo notebook analizza i dati della flotta per rispondere a:
**ci sono evidenze nei dati che giustificano l'uso di Continual Learning (CL)?**

CL e' necessario quando:
- La degradazione non e' uniforme tra impianti → il modello deve adattarsi
  agli impianti degradati senza peggiorare su quelli sani.
- Ci sono recovery post-evento → il modello deve gestire sia regimi
  "degradati" che "recuperati".
- Sottogruppi di impianti evolvono diversamente nel tempo →
  training su un sottogruppo non deve corrompere l'altro.

Se la degradazione fosse uniforme su tutta la flotta, basterebbe
domain adaptation (ricalibrazione globale). Se e' eterogenea, serve CL.

**Input:** output di `analyze_domain_shift_trend.py` (decline_class, level shift,
plant monthly PR, fleet trends).

**Riferimenti:**
- FreeGNN (2025): Continual Source-Free GNN Adaptation for Renewable Energy Forecasting
- ICLR 2025: Continual Domain Adaptation in Time Series via Dual Adapters
- Survey: Graph Learning under Distribution Shifts (2024)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

ROOT = Path('..').resolve()
OUT_TREND = ROOT / 'outputs' / 'domain_shift_trend'
OUT = ROOT / 'outputs' / 'cl_evidence'
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
plant_monthly = pd.read_csv(
    OUT_TREND / 'plant_monthly_performance.csv', parse_dates=['date']
)
real_kwp_trends = pd.read_csv(OUT_TREND / 'real_kwp_plant_pr_trends.csv')
all_trends = pd.read_csv(OUT_TREND / 'all_plant_pr_trends.csv')
all_classes = pd.read_csv(OUT_TREND / 'all_plant_decline_classes.csv')
fleet_monthly = pd.read_csv(
    OUT_TREND / 'fleet_monthly_performance.csv', parse_dates=['date']
)

# Filter plausible PR and edge months
plant_monthly = plant_monthly[
    plant_monthly['pr_pvgis'].between(0.05, 2.0, inclusive='both')
].copy()
edge_mask = pd.Series(False, index=plant_monthly.index)
for _, g in plant_monthly.groupby('plant'):
    if len(g) <= 2:
        edge_mask.loc[g.index] = True
        continue
    si = g.sort_values('date').index
    edge_mask.loc[si[0]] = True
    edge_mask.loc[si[-1]] = True
plant_monthly = plant_monthly[~edge_mask].reset_index(drop=True)

print(f'plant_monthly: {len(plant_monthly)} rows, {plant_monthly["plant"].nunique()} plants')
print(f'date range: {plant_monthly["date"].min()} — {plant_monthly["date"].max()}')
print(f'all_trends: {len(all_trends)} plants')
print(f'decline classes: {all_classes["decline_class"].value_counts().to_dict()}')

## 1. Eterogeneita' della degradazione

Se tutti gli impianti degradano allo stesso ritmo → basta domain adaptation globale
(un unico shift applicato a tutto). Se la degradazione e' eterogenea → sottogruppi
evolvono diversamente → serve CL per non dimenticare pattern dei sottogruppi stabili
mentre ci si adatta a quelli che degradano.

Misure:
- Distribuzione di `relative_change_pct_per_year` e `decline_class`
- Varianza inter-impianto del rate di cambio
- Quanti impianti stabili vs in calo vs in crescita

In [ ]:
rc = all_trends['relative_change_pct_per_year'].dropna()

print('=== Distribuzione relative_change_pct_per_year ===')
print(rc.describe())
print(f'\nIQR: {rc.quantile(0.75) - rc.quantile(0.25):.2f} %/anno')
print(f'Range (5th-95th): {rc.quantile(0.05):.2f} to {rc.quantile(0.95):.2f} %/anno')
print(f'Std: {rc.std():.2f} %/anno')

# Decline class distribution
if 'decline_class' in all_classes.columns:
    dc = all_classes['decline_class'].value_counts()
    print(f'\n=== Decline class distribution ===')
    for cls, cnt in dc.items():
        print(f'  {cls}: {cnt} ({cnt/len(all_classes)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rc, bins=40, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', label='stable (0%/yr)')
axes[0].axvline(rc.median(), color='blue', linestyle='--', label=f'median={rc.median():.1f}%/yr')
axes[0].set_xlabel('Relative change %/year')
axes[0].set_ylabel('N plants')
axes[0].set_title('Degradation rate distribution')
axes[0].legend()

if 'decline_class' in all_classes.columns:
    dc.plot.barh(ax=axes[1], color='steelblue')
    axes[1].set_xlabel('N plants')
    axes[1].set_title('Decline class distribution')

plt.tight_layout()
fig.savefig(OUT / 'degradation_heterogeneity.png', dpi=150, bbox_inches='tight')
plt.show()

# Key metric: coefficient of variation of degradation rate
cv_degradation = rc.std() / abs(rc.mean()) if abs(rc.mean()) > 1e-6 else float('inf')
print(f'\nCV of degradation rate: {cv_degradation:.2f}')
print('CV >> 1 → highly heterogeneous → strong CL case')
print('CV ~ 0 → uniform degradation → DA sufficient')

## 2. Sottogruppi divergenti

Raggruppo impianti per decline_class e confronto le traiettorie PR nel tempo.
Se sottogruppi diversi hanno traiettorie divergenti → il modello deve mantenere
competenza su tutti i regimi simultaneamente (CL), non solo adattarsi all'ultimo.

Se un unico gruppo domina → DA basta.

In [ ]:
# Map plants to decline_class
if 'decline_class' in all_classes.columns:
    plant_class = all_classes.set_index('plant')['decline_class'].to_dict()
    plant_monthly['decline_class'] = plant_monthly['plant'].map(plant_class)

    # Monthly median PR per decline_class
    group_monthly = (
        plant_monthly.groupby(['date', 'decline_class'])['pr_pvgis']
        .median()
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    for cls, g in group_monthly.groupby('decline_class'):
        g = g.sort_values('date')
        ax.plot(g['date'], g['pr_pvgis'], 'o-', label=cls, markersize=4)
    ax.set_xlabel('Month')
    ax.set_ylabel('Median PR_PVGIS')
    ax.set_title('PR trajectories by decline class')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    fig.savefig(OUT / 'subgroup_trajectories.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Divergence: std of group medians per month (increasing = diverging)
    divergence = (
        group_monthly.groupby('date')['pr_pvgis']
        .std()
        .reset_index(name='inter_group_std')
        .sort_values('date')
    )
    print('=== Inter-group divergence over time ===')
    for _, r in divergence.iterrows():
        print(f'  {r["date"].strftime("%Y-%m")}: std={r["inter_group_std"]:.4f}')

    slope_div, _, _, p_div, _ = stats.linregress(
        range(len(divergence)), divergence['inter_group_std']
    )
    print(f'\nDivergence trend: slope={slope_div:.5f}/month, p={p_div:.4f}')
    print('Positive slope = groups diverging over time → CL needed')
    print('Flat/negative = groups move together → DA sufficient')
else:
    print('decline_class not available')

## 3. Recovery post-evento (regime switching)

Se impianti mostrano drop → recovery (PR scende e poi risale), il modello deve
gestire entrambi i regimi. DA pura si adatterebbe al regime "degradato" e poi
sbaglierebbe quando l'impianto recupera. CL mantiene competenza su entrambi.

Cerco: impianti dove PR nella seconda meta' e' PIU' ALTO del break point
(recovery post-level-shift).

In [ ]:
recovery_cols = ['plant', 'plant_id', 'pr_mean', 'best_break_month',
                 'pre_break_mean', 'post_break_mean', 'break_delta_pct',
                 'decline_class', 'monotonic_class']
avail_cols = [c for c in recovery_cols if c in real_kwp_trends.columns]

if 'break_delta_pct' in real_kwp_trends.columns:
    # Plants with positive break_delta (recovery: post > pre)
    recovery_plants = real_kwp_trends[
        real_kwp_trends['break_delta_pct'] > 5.0
    ].sort_values('break_delta_pct', ascending=False)

    decline_plants = real_kwp_trends[
        real_kwp_trends['break_delta_pct'] < -5.0
    ].sort_values('break_delta_pct')

    stable_plants = real_kwp_trends[
        real_kwp_trends['break_delta_pct'].between(-5.0, 5.0)
    ]

    n_total = len(real_kwp_trends)
    print(f'=== Regime switching analysis (real-kWp plants) ===')
    print(f'Total plants: {n_total}')
    print(f'Recovery (break_delta > +5%): {len(recovery_plants)} ({len(recovery_plants)/n_total*100:.1f}%)')
    print(f'Decline (break_delta < -5%):  {len(decline_plants)} ({len(decline_plants)/n_total*100:.1f}%)')
    print(f'Stable (|break_delta| <= 5%): {len(stable_plants)} ({len(stable_plants)/n_total*100:.1f}%)')

    print(f'\nTop 10 recovery plants:')
    display(recovery_plants[avail_cols].head(10))
    print(f'\nTop 10 decline plants:')
    display(decline_plants[avail_cols].head(10))

    # Pie chart
    fig, ax = plt.subplots(figsize=(6, 6))
    sizes = [len(recovery_plants), len(stable_plants), len(decline_plants)]
    labels = [f'Recovery >+5%\n({sizes[0]})', f'Stable\n({sizes[1]})', f'Decline <-5%\n({sizes[2]})']
    colors = ['#4CAF50', '#FFC107', '#F44336']
    ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%')
    ax.set_title('Plant regime classification (break_delta_pct)')
    fig.savefig(OUT / 'regime_switching_pie.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nSe recovery > 0: DA pura fallirebbe (si adatta al drop, poi sbaglia su recovery).')
    print('CL mantiene competenza su entrambi i regimi.')
else:
    print('break_delta_pct not available in real_kwp_trends')

## 4. Concept drift nel residuo per sottogruppi

Calcolo il residuo PR rispetto al fleet median mensile. Se il residuo cambia nel tempo
in modo diverso per sottogruppi diversi → evidence che un modello globale non basta,
serve apprendimento differenziato (CL).

Se il residuo cambia allo stesso modo per tutti → basta ricalibrazione globale (DA).

In [ ]:
# Compute residual: plant PR - fleet median PR for that month
fleet_med = plant_monthly.groupby('date')['pr_pvgis'].median().rename('fleet_median')
pm = plant_monthly.merge(fleet_med, on='date')
pm['residual'] = pm['pr_pvgis'] - pm['fleet_median']

if 'decline_class' in pm.columns:
    # Residual trend per decline_class
    residual_by_group = (
        pm.groupby(['date', 'decline_class'])['residual']
        .median()
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    for cls, g in residual_by_group.groupby('decline_class'):
        g = g.sort_values('date')
        ax.plot(g['date'], g['residual'], 'o-', label=cls, markersize=4)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Month')
    ax.set_ylabel('Median residual (plant PR - fleet median)')
    ax.set_title('Concept drift: residual by decline class over time')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    fig.savefig(OUT / 'concept_drift_residuals.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Test: does residual variance between groups increase over time?
    residual_spread = (
        residual_by_group.groupby('date')['residual']
        .apply(lambda x: x.max() - x.min())
        .reset_index(name='residual_spread')
        .sort_values('date')
    )
    print('=== Residual spread between decline classes ===')
    for _, r in residual_spread.iterrows():
        print(f'  {r["date"].strftime("%Y-%m")}: spread={r["residual_spread"]:.4f}')

    slope_res, _, _, p_res, _ = stats.linregress(
        range(len(residual_spread)), residual_spread['residual_spread']
    )
    print(f'\nSpread trend: slope={slope_res:.5f}/month, p={p_res:.4f}')
    print('Positive slope = groups diverge in residuals → CL needed')
    print('Flat = groups shift together → DA sufficient')

## 5. Coesistenza di regimi multipli nello stesso periodo

CL e' necessario quando il modello deve predire simultaneamente impianti in regimi
diversi. Se in ogni mese ci sono sia impianti "stabili" che "degradati" che "in recovery",
il modello non puo' semplicemente shiftare globalmente — deve mantenere competenza
su tutti i regimi.

Misura: distribuzione intra-mese di PR per diversi sottogruppi.

In [ ]:
if 'decline_class' in pm.columns:
    # Per-month distribution width by decline class
    monthly_stats = (
        pm.groupby(['date', 'decline_class'])['pr_pvgis']
        .agg(['median', 'std', 'count'])
        .reset_index()
    )

    # Box plot: PR distribution per month, colored by decline_class
    months = sorted(pm['date'].unique())
    major_classes = pm['decline_class'].value_counts().head(4).index.tolist()
    pm_major = pm[pm['decline_class'].isin(major_classes)]

    fig, ax = plt.subplots(figsize=(14, 6))
    positions = []
    labels = []
    colors_map = dict(zip(major_classes, ['#4CAF50', '#FFC107', '#F44336', '#2196F3']))

    for i, month in enumerate(months):
        month_data = pm_major[pm_major['date'] == month]
        for j, cls in enumerate(major_classes):
            cls_data = month_data[month_data['decline_class'] == cls]['pr_pvgis'].dropna()
            if len(cls_data) < 3:
                continue
            pos = i * (len(major_classes) + 1) + j
            bp = ax.boxplot(
                [cls_data.values], positions=[pos], widths=0.6,
                patch_artist=True, showfliers=False
            )
            bp['boxes'][0].set_facecolor(colors_map[cls])
            bp['boxes'][0].set_alpha(0.6)

    # Custom legend and x-axis
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=colors_map[c], alpha=0.6, label=c) for c in major_classes]
    ax.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
    tick_pos = [i * (len(major_classes) + 1) + len(major_classes) / 2 - 0.5 for i in range(len(months))]
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([m.strftime('%Y-%m') for m in months], rotation=45)
    ax.set_ylabel('PR_PVGIS')
    ax.set_title('PR distribution per month by decline class')
    plt.tight_layout()
    fig.savefig(OUT / 'regime_coexistence_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Overlap measure: how much do distributions overlap?
    print('=== Distribution overlap between major classes ===')
    for month in months:
        md = pm_major[pm_major['date'] == month]
        medians = md.groupby('decline_class')['pr_pvgis'].median()
        if len(medians) >= 2:
            spread = medians.max() - medians.min()
            pooled_std = md.groupby('decline_class')['pr_pvgis'].std().mean()
            sep_ratio = spread / pooled_std if pooled_std > 1e-6 else 0
            print(f'  {month.strftime("%Y-%m")}: median_spread={spread:.4f}, '
                  f'pooled_std={pooled_std:.4f}, separation_ratio={sep_ratio:.2f}')
    print('\nseparation_ratio > 1 = distinct regimes coexist → CL needed')
    print('separation_ratio < 0.5 = heavily overlapping → DA may suffice')

## 6. Sample reliability scoring

Complemento: rileva eventi puntuali anomali di produzione (Z-score, MoM shock)
e classifica per `action_for_training` (include/downweight/exclude).
Utile come pre-processing sia per DA che per CL.

Soglie basate su letteratura NREL:
- Drop <15%: ambientale plausibile → include
- Drop 15-25%: ambiguo → downweight (o include se fleet-wide)
- Drop >25%: equipment/data issue → exclude

In [ ]:
MODERATE_DROP_PCT = 15.0
SEVERE_DROP_PCT = 25.0
ZSCORE_THRESHOLD = 2.0
MOM_DROP_PCT = -15.0
MOM_SPIKE_PCT = 30.0
COOCCURRENCE_PCT = 0.05
n_fleet = plant_monthly['plant'].nunique()
COOCCURRENCE_MIN = max(5, int(n_fleet * COOCCURRENCE_PCT))


def modified_zscore(series):
    med = series.median()
    mad = np.median(np.abs(series - med))
    if mad < 1e-9:
        return pd.Series(0.0, index=series.index)
    return 0.6745 * (series - med) / mad


# Z-score anomalies
zscore_records = []
for (plant, plant_id), g in plant_monthly.groupby(['plant', 'plant_id']):
    pr = g.set_index('date')['pr_pvgis'].dropna().sort_index()
    if len(pr) < 4:
        continue
    z = modified_zscore(pr)
    for dt, zval in z[z.abs() > ZSCORE_THRESHOLD].items():
        zscore_records.append({
            'plant': plant, 'plant_id': plant_id, 'date': dt,
            'pr_pvgis': pr.loc[dt], 'reference': pr.median(),
            'severity': abs(zval),
            'event_type': 'drop' if zval < 0 else 'spike',
            'method': 'modified_zscore',
        })

# MoM shocks
mom_records = []
for (plant, plant_id), g in plant_monthly.groupby(['plant', 'plant_id']):
    pr = g.set_index('date')['pr_pvgis'].dropna().sort_index()
    if len(pr) < 3:
        continue
    pct = pr.pct_change() * 100
    for i in range(1, len(pr)):
        chg = pct.iloc[i]
        if pd.isna(chg) or (chg > MOM_DROP_PCT and chg < MOM_SPIKE_PCT):
            continue
        recovery = False
        if i + 1 < len(pct):
            nxt = pct.iloc[i + 1]
            if not pd.isna(nxt):
                if chg < MOM_DROP_PCT and nxt > abs(chg) * 0.5:
                    recovery = True
                elif chg > MOM_SPIKE_PCT and nxt < -chg * 0.5:
                    recovery = True
        etype = ('transient_drop' if (chg < 0 and recovery) else
                 'transient_spike' if (chg > 0 and recovery) else
                 'sustained_drop' if chg < 0 else 'sustained_spike')
        mom_records.append({
            'plant': plant, 'plant_id': plant_id, 'date': pr.index[i],
            'pr_pvgis': pr.iloc[i], 'reference': pr.iloc[i - 1],
            'severity': abs(chg),
            'event_type': etype, 'method': 'mom_shock',
        })

# Fleet co-occurrence
all_events = pd.concat(
    [pd.DataFrame(zscore_records), pd.DataFrame(mom_records)], ignore_index=True
)
fleet_event_months = set()
if not all_events.empty:
    co = all_events.groupby('date')['plant'].nunique().reset_index(name='n')
    fleet_event_months = set(co[co['n'] >= COOCCURRENCE_MIN]['date'])

# Classify
def classify(row):
    is_fleet = row['date'] in fleet_event_months
    ref = row['reference']
    if ref > 0:
        dev = abs((row['pr_pvgis'] - ref) / ref) * 100
    else:
        dev = row['severity']
    if dev < MODERATE_DROP_PCT:
        return 'include'
    elif dev < SEVERE_DROP_PCT:
        return 'include' if is_fleet else 'downweight'
    else:
        return 'downweight' if is_fleet else 'exclude'

if not all_events.empty:
    all_events['action_for_training'] = all_events.apply(classify, axis=1)
    all_events.to_csv(OUT / 'event_catalog.csv', index=False)
    print(f'Events: {len(all_events)}, plants: {all_events["plant"].nunique()}')
    print(f'Fleet event months (>={COOCCURRENCE_MIN} plants): {len(fleet_event_months)}')
    print(f'\naction_for_training:')
    for a, c in all_events['action_for_training'].value_counts().items():
        print(f'  {a}: {c} ({c/len(all_events)*100:.1f}%)')
    print(f'\nBy method:')
    print(all_events.groupby(['method', 'action_for_training']).size().unstack(fill_value=0))
else:
    print('No events detected.')

## Report compatto

Copia l'output per analisi esterna.

In [ ]:
_L = []
_L.append('=' * 70)
_L.append('CL EVIDENCE ANALYSIS — COMPACT REPORT')
_L.append('=' * 70)
_L.append(f'Data: {plant_monthly["plant"].nunique()} plants, '
          f'{plant_monthly["date"].min().strftime("%Y-%m")} to '
          f'{plant_monthly["date"].max().strftime("%Y-%m")}')

# 1. Heterogeneity
_L.append('\n' + '=' * 70)
_L.append('1. DEGRADATION HETEROGENEITY')
_L.append('=' * 70)
rc = all_trends['relative_change_pct_per_year'].dropna()
_L.append(f'N plants: {len(rc)}')
_L.append(f'Mean: {rc.mean():.2f} %/yr, Median: {rc.median():.2f} %/yr')
_L.append(f'Std: {rc.std():.2f} %/yr')
_L.append(f'IQR: {rc.quantile(0.75) - rc.quantile(0.25):.2f} %/yr')
_L.append(f'Range (5th-95th): {rc.quantile(0.05):.2f} to {rc.quantile(0.95):.2f}')
cv = rc.std() / abs(rc.mean()) if abs(rc.mean()) > 1e-6 else float('inf')
_L.append(f'CV: {cv:.2f} (>>1 = heterogeneous → CL, ~0 = uniform → DA)')
if 'decline_class' in all_classes.columns:
    _L.append('Decline classes:')
    for cls, cnt in all_classes['decline_class'].value_counts().items():
        _L.append(f'  {cls}: {cnt} ({cnt/len(all_classes)*100:.1f}%)')

# 2. Subgroups
_L.append('\n' + '=' * 70)
_L.append('2. SUBGROUP DIVERGENCE')
_L.append('=' * 70)
if 'decline_class' in pm.columns:
    grp = pm.groupby(['date', 'decline_class'])['pr_pvgis'].median().reset_index()
    div = grp.groupby('date')['pr_pvgis'].std().reset_index(name='std').sort_values('date')
    for _, r in div.iterrows():
        _L.append(f'  {r["date"].strftime("%Y-%m")}: inter-group std={r["std"]:.4f}')
    sl, _, _, pv, _ = stats.linregress(range(len(div)), div['std'])
    _L.append(f'Divergence trend: slope={sl:.5f}/month, p={pv:.4f}')
    _L.append('Positive slope → groups diverge → CL. Flat → DA.')

# 3. Recovery
_L.append('\n' + '=' * 70)
_L.append('3. REGIME SWITCHING (recovery vs decline)')
_L.append('=' * 70)
if 'break_delta_pct' in real_kwp_trends.columns:
    bd = real_kwp_trends['break_delta_pct']
    n_rec = (bd > 5).sum()
    n_dec = (bd < -5).sum()
    n_stab = bd.between(-5, 5).sum()
    n_tot = len(bd)
    _L.append(f'Recovery (>+5%): {n_rec} ({n_rec/n_tot*100:.1f}%)')
    _L.append(f'Stable (±5%):   {n_stab} ({n_stab/n_tot*100:.1f}%)')
    _L.append(f'Decline (<-5%): {n_dec} ({n_dec/n_tot*100:.1f}%)')
    _L.append('Recovery > 0 → regimes coexist → DA alone fails → CL needed.')

# 4. Concept drift
_L.append('\n' + '=' * 70)
_L.append('4. CONCEPT DRIFT IN RESIDUALS')
_L.append('=' * 70)
if 'decline_class' in pm.columns:
    res_grp = pm.groupby(['date', 'decline_class'])['residual'].median().reset_index()
    res_spread = res_grp.groupby('date')['residual'].apply(lambda x: x.max() - x.min()).reset_index(name='spread').sort_values('date')
    for _, r in res_spread.iterrows():
        _L.append(f'  {r["date"].strftime("%Y-%m")}: residual_spread={r["spread"]:.4f}')
    sl2, _, _, pv2, _ = stats.linregress(range(len(res_spread)), res_spread['spread'])
    _L.append(f'Spread trend: slope={sl2:.5f}/month, p={pv2:.4f}')

# 5. Sample reliability
_L.append('\n' + '=' * 70)
_L.append('5. SAMPLE RELIABILITY (event catalog)')
_L.append('=' * 70)
if not all_events.empty:
    _L.append(f'Total events: {len(all_events)}')
    for a, c in all_events['action_for_training'].value_counts().items():
        _L.append(f'  {a}: {c} ({c/len(all_events)*100:.1f}%)')
    _L.append(f'Fleet event months: {len(fleet_event_months)}')
    _L.append(all_events.groupby(['method', 'action_for_training']).size().unstack(fill_value=0).to_string())

# Conclusion
_L.append('\n' + '=' * 70)
_L.append('CONCLUSION')
_L.append('=' * 70)
_L.append(
    'CL is justified when:\n'
    '  1. Degradation is heterogeneous (CV >> 1)\n'
    '  2. Subgroups diverge over time (positive divergence slope)\n'
    '  3. Recovery exists (model must handle multiple regimes)\n'
    '  4. Residual spread increases (concept drift is group-specific)\n'
    '\n'
    'DA alone suffices when:\n'
    '  1. All plants shift uniformly (CV ~ 0)\n'
    '  2. No recovery (monotonic shift)\n'
    '  3. Residuals shift together (global recalibration works)\n'
    '\n'
    'CDA (Continual Domain Adaptation) is the recommended approach when\n'
    'both DA and CL evidence are present — distribution shifts AND\n'
    'heterogeneous evolution require both adaptation and memory retention.\n'
    '\n'
    'References:\n'
    '  - FreeGNN (2025): Continual Source-Free GNN Adaptation\n'
    '  - ICLR 2025: Continual DA via Dual Adapters\n'
    '  - Survey: Graph Learning under Distribution Shifts (2024)'
)

print('\n'.join(_L))

## Lettura per la tesi

Questo notebook fornisce evidenze quantitative per giustificare l'uso di
Continual Learning (o Continual Domain Adaptation) nel forecasting PV.

**Evidenza 1 — Eterogeneita'.** Se il CV della degradation rate e' alto,
impianti diversi si comportano diversamente. Un modello che fa DA globale
penalizza gli impianti stabili per adattarsi a quelli degradati.

**Evidenza 2 — Divergenza sottogruppi.** Se la std inter-gruppo cresce
nel tempo, i sottogruppi si stanno separando. CL mantiene competenza
su tutti.

**Evidenza 3 — Recovery.** Se esistono impianti che droppano e recuperano,
DA pura fallirebbe (si adatta al drop, poi sbaglia sul recovery). CL
mantiene entrambi i regimi.

**Evidenza 4 — Concept drift differenziato.** Se il residuo (plant PR - fleet)
cambia in modo diverso per gruppi diversi, serve apprendimento differenziato.

**Sample reliability.** L'event catalog con `action_for_training` serve come
pre-processing per qualsiasi strategia (DA, CL, o CDA).